### Neural Network from Scratch


In [1]:
from abc import abstractmethod
import pandas as pd
import numpy as np

# train test split
from sklearn.model_selection import train_test_split

### Data split


In [ ]:
df = pd.read_csv("data_logistic.csv", header=None)

In [3]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Không có đoạn reshape này thì sẽ bị lỗi
X_train = X_train.reshape(-1, 1, 2)
y_train = y_train.reshape(-1, 1, 1)
X_test = X_test.reshape(-1, 1, 2)
y_test = y_test.reshape(-1, 1, 1)

In [6]:
class Layer:
    def __init__(self, input, output, input_shape, output_shape):

        self.input = input
        self.output = output
        self.input_shape = input_shape
        self.output_shape = output_shape

    @abstractmethod
    def output(self):
        return self.output

    @abstractmethod
    def input(self):
        return self.input

    @abstractmethod
    def input_shape(self):
        return self.input_shape

    @abstractmethod
    def output_shape(self):
        return self.output_shape

    @abstractmethod
    def forward_propagation(self):
        pass

    @abstractmethod
    def backward_propagation(self):
        pass

### Tạo lớp Fully connected layer


In [7]:
class FCLayer(Layer):
    def __init__(self, input_shape, output_shape):
        self.input_shape = input_shape
        self.output_shape = output_shape
        self.weights = np.random.rand(input_shape, output_shape) - 0.5
        self.bias = np.random.rand(1, output_shape) - 0.5

    def forward_propagation(self, input):
        self.input = input
        self.output = np.dot(self.input, self.weights) + self.bias
        return self.output

    def backward_propagation(self, output_error, learning_rate):
        input_error = np.dot(output_error, self.weights.T)
        weights_error = np.dot(self.input.T, output_error)
        self.weights -= learning_rate * weights_error
        self.bias -= learning_rate * output_error
        return input_error

In [8]:
class ActivationLayer(Layer):
    def __init__(self, input_shape, output_shape, activation, activation_prime):
        self.input_shape = input_shape
        self.output_shape = output_shape
        self.activation = activation
        self.activation_prime = activation_prime

    def forward_propagation(self, input):
        self.input = input
        self.output = self.activation(self.input)
        return self.output

    def backward_propagation(self, output_error, learning_rate):
        return self.activation_prime(self.input) * output_error

In [9]:
class Network:
    def __init__(self):
        self.layers = []
        self.loss = None
        self.loss_prime = None

    def add(self, layer):
        self.layers.append(layer)

    def setup_loss(self, loss, loss_prime):
        self.loss = loss
        self.loss_prime = loss_prime

    def predict(self, input):
        result = []
        n = len(input)

        # Dự đoán cho từng mẫu dữ liệu
        for i in range(n):
            output = input[i]
            for layer in self.layers:
                output = layer.forward_propagation(output)
            result.append(output)
        return result

    def fit(self, X_train, y_train, epochs, learning_rate):
        n = len(X_train)

        # Huấn luyện cho từng epoch
        for i in range(epochs):
            err = 0
            for j in range(n):
                output = X_train[j]

                for layer in self.layers:
                    output = layer.forward_propagation(output)

                err += self.loss(y_train[j], output)

                # Lan truyền ngược
                error = self.loss_prime(y_train[j], output)

                for layer in reversed(self.layers):
                    error = layer.backward_propagation(error, learning_rate)

            err /= n
            print("epoch %d/%d   error=%f" % (i + 1, epochs, err))

### Xây dựng kiến trúc mạng


In [ ]:
def relu(z):
    return np.maximum(0, z)


def relu_prime(z):
    return (z > 0).astype(int)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def sigmoid_prime(z):
    return sigmoid(z) * (1 - sigmoid(z))

In [11]:
def loss(y_true, y_pred):
    return np.mean(np.power(y_true - y_pred, 2))


def loss_prime(y_true, y_pred):
    return 2 * (y_pred - y_true) / np.size(y_true)

In [12]:
input_shape = X_train.shape[1]
output_shape = 1

In [13]:
# input, output là theo data, còn ở hidden layer có bao nhiêu neural là tuỳ mùnh

network = Network()
network.add(FCLayer(2, 4))  # network.add(FCLayer((1,2),(1,3|4|5...))) cũng được
network.add(
    ActivationLayer((1, 4), (1, 4), relu, relu_prime)
)  # network.add(FCLayer((1,5),(1,5))) cũng được nếu ở trên chọn 5 neural ở hidden layer đầu
network.add(FCLayer(4, 1))
network.add(ActivationLayer((1, 1), (1, 1), sigmoid, sigmoid_prime))

In [14]:
network.setup_loss(loss, loss_prime)
network.fit(X_train, y_train, epochs=1000, learning_rate=0.01)

epoch 1/1000   error=0.321796
epoch 2/1000   error=0.256828
epoch 3/1000   error=0.250604
epoch 4/1000   error=0.247656
epoch 5/1000   error=0.246651
epoch 6/1000   error=0.245931
epoch 7/1000   error=0.245057
epoch 8/1000   error=0.244230
epoch 9/1000   error=0.243654
epoch 10/1000   error=0.243203
epoch 11/1000   error=0.242760
epoch 12/1000   error=0.242232
epoch 13/1000   error=0.241800
epoch 14/1000   error=0.241218
epoch 15/1000   error=0.240722
epoch 16/1000   error=0.240112
epoch 17/1000   error=0.239576
epoch 18/1000   error=0.238892
epoch 19/1000   error=0.238335
epoch 20/1000   error=0.237740
epoch 21/1000   error=0.237066
epoch 22/1000   error=0.236446
epoch 23/1000   error=0.235887
epoch 24/1000   error=0.235208
epoch 25/1000   error=0.234586
epoch 26/1000   error=0.234023
epoch 27/1000   error=0.233390
epoch 28/1000   error=0.232834
epoch 29/1000   error=0.232200
epoch 30/1000   error=0.231645
epoch 31/1000   error=0.231061
epoch 32/1000   error=0.230451
epoch 33/1000   e

In [15]:
def mapping_output(output):
    if output >= 0.5:
        return 1
    else:
        return 0

In [ ]:
output = network.predict(X_test)
output

[array([[0.04270471]]),
 array([[0.93660726]]),
 array([[0.00393836]]),
 array([[0.00116653]]),
 array([[0.04680093]]),
 array([[0.22110995]]),
 array([[0.93660726]]),
 array([[0.93660726]]),
 array([[0.93660726]]),
 array([[0.93660726]]),
 array([[0.81792219]]),
 array([[0.93660726]]),
 array([[0.14795787]]),
 array([[0.93660726]]),
 array([[0.03309551]]),
 array([[0.93660726]]),
 array([[0.11702663]]),
 array([[0.7803857]]),
 array([[0.34046328]]),
 array([[0.01143192]])]

In [20]:
for i in range(len(X_test)):
    print(f"Input: {X_test[i]}, Predicted: {output[i]}, Actual: {y_test[i]}")

Input: [[2.23818348 4.5348437 ]], Predicted: [[0.04270471]], Actual: [[0]]
Input: [[7.75053006 9.05717188]], Predicted: [[0.93660726]], Actual: [[1]]
Input: [[1.13248839 3.26491181]], Predicted: [[0.00393836]], Actual: [[0]]
Input: [[0.45238867 2.77469425]], Predicted: [[0.00116653]], Actual: [[0]]
Input: [[0.09805288 7.21451254]], Predicted: [[0.04680093]], Actual: [[0]]
Input: [[4.53017137 3.761759  ]], Predicted: [[0.22110995]], Actual: [[0]]
Input: [[8.83477098 3.15220412]], Predicted: [[0.93660726]], Actual: [[1]]
Input: [[9.00037648 9.54932786]], Predicted: [[0.93660726]], Actual: [[1]]
Input: [[6.86140497 9.65530972]], Predicted: [[0.93660726]], Actual: [[1]]
Input: [[4.85506424 9.63996157]], Predicted: [[0.93660726]], Actual: [[1]]
Input: [[4.54033196 6.71387336]], Predicted: [[0.81792219]], Actual: [[1]]
Input: [[7.15470123 8.0147902 ]], Predicted: [[0.93660726]], Actual: [[1]]
Input: [[6.50581749 0.86310223]], Predicted: [[0.14795787]], Actual: [[0]]
Input: [[6.33309623 7.240